# imports
this notebook is to synthesize English-like renditions of Irish from the English-like transcriptions 

In [1]:
from datasets import load_from_disk

/home/peter/miniconda3/envs/thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

# Walk up from the notebook's location to find the project root
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / ".git").exists(): # anchor project root to .git
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "fleurs" / "synthetic"

# Load data

In [3]:
dataset_dict = load_from_disk(DATA_PATH)

Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Getting issues with specific items. what is causing this?

In [4]:
dataset_dict[1]

{'audio': {'path': 'spalp.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(21504,)),
  'sampling_rate': 16000},
 'phonetic': 'sˠ pˠ o l̻ˠ pˠ',
 'English ASR transcriptions': 's p ɑ l p'}

Apparently ɑ is only presently accepted by google's english tts in its long form ɑː

In [5]:
all_chars = set()
for example in dataset_dict:
    all_chars.update(example["English ASR transcriptions"].replace(" ", ""))
print(all_chars - set("pbtkdgmnŋfvszθðʃʒhlɹʧʤjwæɑəɚɛɪɔʊʌ") - set("ːaɪʊeɔoU"))

{'̥', 'ɨ', 'i', 'ɾ', 'ʔ', 'ʉ', 'u', 'ɦ', 'ɝ', '̩', '̃'}


that's not the only one apparently.

## prepare data

some phonemes are unsupported by google tts, so we will normalize them to the closest equivalent, which is in most cases is lengthening vowels or removing diacritics.

In [6]:
'''ipa_normalise = {
    "ɑ": "ɑː",   # short open back → long (closest supported)
    "i": "iː",   # short close front → long
    "u": "uː",   # short close back → long
    "ɔ": "ɔː",   # short open-mid back → long (obs! this is present in dipthongs)
    "ə̥": "ə",    # devoiced schwa → plain schwa
    "ɾ̃": "ɹ",    # nasalized alveolar flap → plain retroflex
    "ɦ": "h",    # voiced glottal fricative → unvoiced
    "ʉ": "ʊ",    # slightly backed + open 
    "ʔ": "g",    # glottal stop → velar stop
    "ɝ": "ə",    # rhotic open-mid medial → plain
    "ɨ": "ɪ",    # slightly fronted + opened
    "ɾ": "ɹ",    # alveolar tap → approximate
}
ipa = 's p ɑ l p'
tokens = ipa.split()
tokens = [ipa_normalise.get(token, token) for token in tokens] # just return token if it isn't in normalization dict
print(tokens)'''

'ipa_normalise = {\n    "ɑ": "ɑː",   # short open back → long (closest supported)\n    "i": "iː",   # short close front → long\n    "u": "uː",   # short close back → long\n    "ɔ": "ɔː",   # short open-mid back → long (obs! this is present in dipthongs)\n    "ə̥": "ə",    # devoiced schwa → plain schwa\n    "ɾ̃": "ɹ",    # nasalized alveolar flap → plain retroflex\n    "ɦ": "h",    # voiced glottal fricative → unvoiced\n    "ʉ": "ʊ",    # slightly backed + open \n    "ʔ": "g",    # glottal stop → velar stop\n    "ɝ": "ə",    # rhotic open-mid medial → plain\n    "ɨ": "ɪ",    # slightly fronted + opened\n    "ɾ": "ɹ",    # alveolar tap → approximate\n}\nipa = \'s p ɑ l p\'\ntokens = ipa.split()\ntokens = [ipa_normalise.get(token, token) for token in tokens] # just return token if it isn\'t in normalization dict\nprint(tokens)'

In [7]:
ipa_normalise = {}

forgot to strip spaces between spaces between phones. can strip those as well before synthesis

In [8]:
import re

def normalise_ipa(example):
    ipa = example['English ASR transcriptions']
    tokens = ipa.split()
    example['English ASR transcriptions'] = "".join([ipa_normalise.get(token, token) for token in tokens])
    return example

dataset_dict = dataset_dict.map(normalise_ipa)

Map: 100%|██████████| 19136/19136 [00:03<00:00, 4916.61 examples/s]


In [15]:
dataset_dict[1]

{'audio': {'path': 'spalp.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(21504,)),
  'sampling_rate': 16000},
 'phonetic': 'sˠ pˠ o l̻ˠ pˠ',
 'English ASR transcriptions': 'spɑlp'}

# Synthesize examples

In [10]:
from google.cloud import texttospeech
import random
import time

client = texttospeech.TextToSpeechClient()

# fetch and shuffle voices once
voices = client.list_voices(language_code="en-US").voices
en_us_voices = [
    v.name for v in voices
    if "en-US" in v.language_codes
    and "Chirp3" in v.name
]
random.seed(42)
shuffled_voices = random.sample(en_us_voices, len(en_us_voices))

output_dir = PROJECT_ROOT / "data" / "fleurs" / "synthetic_audio"
output_dir.mkdir(parents=True, exist_ok=True)

In [11]:
shuffled_voices

['en-US-Chirp3-HD-Pulcherrima',
 'en-US-Chirp3-HD-Algieba',
 'en-US-Chirp3-HD-Achernar',
 'en-US-Chirp3-HD-Sadaltager',
 'en-US-Chirp3-HD-Charon',
 'en-US-Chirp3-HD-Callirrhoe',
 'en-US-Chirp3-HD-Schedar',
 'en-US-Chirp3-HD-Alnilam',
 'en-US-Chirp3-HD-Zephyr',
 'en-US-Chirp3-HD-Leda',
 'en-US-Chirp3-HD-Algenib',
 'en-US-Chirp3-HD-Orus',
 'en-US-Chirp3-HD-Gacrux',
 'en-US-Chirp3-HD-Achird',
 'en-US-Chirp3-HD-Vindemiatrix',
 'en-US-Chirp3-HD-Laomedeia',
 'en-US-Chirp3-HD-Rasalgethi',
 'en-US-Chirp3-HD-Zubenelgenubi',
 'en-US-Chirp3-HD-Sulafat',
 'en-US-Chirp3-HD-Despina',
 'en-US-Chirp3-HD-Kore',
 'en-US-Chirp3-HD-Erinome',
 'en-US-Chirp3-HD-Fenrir',
 'en-US-Chirp3-HD-Aoede',
 'en-US-Chirp3-HD-Autonoe',
 'en-US-Chirp3-HD-Sadachbia',
 'en-US-Chirp3-HD-Umbriel',
 'en-US-Chirp3-HD-Enceladus',
 'en-US-Chirp3-HD-Iapetus',
 'en-US-Chirp3-HD-Puck']

In [19]:
def synthesize_example(example, idx):
    output_path = output_dir / f"{idx}.mp3"

    if output_path.exists():
        example["synthetic_audio_path"] = str(output_path)
        return example

    ipa = example["English ASR transcriptions"]
    ssml = f'<speak><phoneme alphabet="ipa" ph="{ipa}">word</phoneme></speak>'

    voice = texttospeech.VoiceSelectionParams(
        language_code="en-US",
        name=shuffled_voices[idx % len(shuffled_voices)]
    )
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )
    
    for attempt in range(3): # exponential backoff for 3 retries
        try:
            response = client.synthesize_speech(
                input=texttospeech.SynthesisInput(ssml=ssml),
                voice=voice,
                audio_config=audio_config
            )
            with open(output_path, "wb") as f:
                f.write(response.audio_content)
            example["synthetic_audio_path"] = str(output_path)
            time.sleep(0.1)
            return example
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)  # 1s, then 2s
            else:
                print(f"Failed on idx {idx} after 3 attempts: {e}")
                example["synthetic_audio_path"] = ""
                return example
    

In [16]:
example = dataset_dict[1]
synthesize_example(example,1)

<speak><phoneme alphabet="ipa" ph="spɑlp">word</phoneme></speak>


{'audio': {'path': 'spalp.wav',
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(21504,)),
  'sampling_rate': 16000},
 'phonetic': 'sˠ pˠ o l̻ˠ pˠ',
 'English ASR transcriptions': 'spɑlp',
 'synthetic_audio_path': '/home/peter/Desktop/thesis/ThesisProject/data/fleurs/synthetic_audio/1.mp3'}

# Batch synthesis and save

In [ ]:
dataset_dict = dataset_dict.map(synthesize_example, with_indices=True, load_from_cache_file=False)


Map: 100%|██████████| 19136/19136 [3:43:46<00:00,  1.43 examples/s]  


PermissionError: Tried to overwrite /home/peter/Desktop/thesis/ThesisProject/data/fleurs/synthetic but a dataset can't overwrite itself.

In [21]:
dataset_dict.save_to_disk(str(PROJECT_ROOT / "data" / "fleurs" / "synthetic" / "new"))

Saving the dataset (2/2 shards): 100%|██████████| 19136/19136 [00:01<00:00, 15971.41 examples/s]
